# 02 — Risk Stratification Model
XGBoost to identify High/Critical priority patients for proactive outreach.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import sys; sys.path.append("../src")
from risk_model import (load_data, make_binary_target, build_model,
                        cross_validate, evaluate, save_model,
                        score_panel, generate_weekly_worklist,
                        NUMERIC_FEATURES, CATEGORICAL_FEATURES)

df = load_data("../data/raw/patient_panel_flat.csv")
print(f"Panel: {len(df):,} patients")


## Prepare features and target

In [ ]:
available_num = [c for c in NUMERIC_FEATURES if c in df.columns]
X = df[available_num + CATEGORICAL_FEATURES]
y = make_binary_target(df)

print(f"Target distribution:")
print(y.value_counts().to_string())
print(f"High/Critical rate: {y.mean():.1%}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")


## Cross-validation

In [ ]:
model = build_model()
print("Running 5-fold cross-validation...")
cv_results = cross_validate(model, X_train, y_train)


## Train and evaluate on test set

In [ ]:
model.fit(X_train, y_train)
results = evaluate(model, X_test, y_test)


## ROC and Precision-Recall curves

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
RocCurveDisplay.from_estimator(model, X_test, y_test, ax=axes[0], name="XGBoost Panel Risk Model")
axes[0].set_title("ROC Curve — test set", fontsize=13)
PrecisionRecallDisplay.from_estimator(model, X_test, y_test, ax=axes[1], name="XGBoost Panel Risk Model")
axes[1].set_title("Precision-Recall Curve — test set", fontsize=13)
plt.tight_layout()
plt.savefig("../reports/figures/model_performance.png", dpi=150, bbox_inches="tight")
plt.show()


## SHAP feature importance

In [ ]:
import shap

preprocessor = model.named_steps["preprocessor"]
classifier = model.named_steps["classifier"]
X_test_transformed = preprocessor.transform(X_test)
from risk_model import get_feature_names
feature_names = get_feature_names(model)

explainer = shap.TreeExplainer(classifier)
shap_values = explainer.shap_values(X_test_transformed)

shap.summary_plot(shap_values, X_test_transformed,
                  feature_names=feature_names, max_display=15, show=False)
plt.title("SHAP feature importance — top 15 predictors")
plt.tight_layout()
plt.savefig("../reports/figures/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()


## Generate weekly worklist

In [ ]:
save_model(model, "../models/risk_model.joblib")

scored = score_panel(model, df)
worklist = generate_weekly_worklist(scored, n_patients=15)

print("This week's proactive outreach list (top 15 patients):")
print(worklist[["patient_id", "age", "n_conditions", "n_quality_gaps",
                "days_since_pcp_visit", "er_visits_12m",
                "risk_probability", "risk_tier"]].to_string())

worklist.to_csv("../data/processed/weekly_worklist.csv")
scored.to_csv("../data/processed/scored_panel.csv", index=False)
print("Saved.")


## Clinical interpretation

**Days since last PCP visit** is the strongest predictor. Patients lost to follow-up for 180+ days in a chronic disease panel are accumulating unmanaged risk silently.

**Number of quality gaps** compounds risk. A patient overdue for HbA1c, eye exam, and kidney screening is missing three clinical safety nets simultaneously — not just three checkboxes.

**Prior hospitalization** is the strongest published predictor of future hospitalization. The model weights this correctly.

**SDOH factors** elevate risk independent of clinical signals. A patient who can't afford food is unlikely to afford medications — the model captures this structural reality.

The weekly worklist reduces 800 patients to 15 actionable outreach targets. The PCP reviews the list in 10–15 minutes, delegates to a care coordinator, and the clinical team focuses energy where it will have the most impact.
